# Notebook 02 — Preprocessing
Runs the full preprocessing pipeline and verifies the output splits.

In [ ]:
import subprocess, os
from pathlib import Path

REPO_URL  = 'https://github.com/calvinkatoroy/tugas-akhir-ai-kel-08.git'
REPO_NAME = 'tugas-akhir-ai-kel-08'

cwd = Path.cwd()

if (cwd / '../src').resolve().exists():
    result = subprocess.run(['git', '-C', str((cwd / '..').resolve()), 'pull'],
                            capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')

elif (cwd / 'src').exists():
    result = subprocess.run(['git', 'pull'], capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')
    os.chdir('notebooks')

else:
    repo_path = cwd / REPO_NAME
    if not repo_path.exists():
        print(f'Cloning {REPO_URL} ...')
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    else:
        print('Repo found, pulling latest...')
        subprocess.run(['git', '-C', str(repo_path), 'pull'], capture_output=True)
    os.chdir(repo_path / 'notebooks')

print(f'Working dir: {Path.cwd()}')

In [ ]:
import sys
sys.path.insert(0, '..')

import random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
from src.preprocessing import run_preprocessing

info = run_preprocessing(
    config_path='../config.yaml',
    kaggle_dir=KAGGLE_DIR,
    splits_dir=SPLITS_DIR,
)
print('\nSummary:', info)

In [ ]:
# Verify saved splits
import yaml, joblib

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

# splits_dir already set by Drive mount cell above
X_train     = np.load(SPLITS_DIR / 'X_train.npy')
X_val       = np.load(SPLITS_DIR / 'X_val.npy')
X_test      = np.load(SPLITS_DIR / 'X_test.npy')
y_train     = np.load(SPLITS_DIR / 'y_train.npy')
y_val       = np.load(SPLITS_DIR / 'y_val.npy')
y_test      = np.load(SPLITS_DIR / 'y_test.npy')
X_train_seq = np.load(SPLITS_DIR / 'X_train_seq.npy')
X_val_seq   = np.load(SPLITS_DIR / 'X_val_seq.npy')
X_test_seq  = np.load(SPLITS_DIR / 'X_test_seq.npy')
y_train_seq = np.load(SPLITS_DIR / 'y_train_seq.npy')
y_val_seq   = np.load(SPLITS_DIR / 'y_val_seq.npy')
y_test_seq  = np.load(SPLITS_DIR / 'y_test_seq.npy')

print('=== Flat splits (for Random Forest) ===')
print(f'  X_train:  {X_train.shape}  y_train: {y_train.shape}')
print(f'  X_val:    {X_val.shape}    y_val:   {y_val.shape}')
print(f'  X_test:   {X_test.shape}   y_test:  {y_test.shape}')
print('\n=== Sequence splits (for LSTM/GRU) ===')
print(f'  X_train_seq: {X_train_seq.shape}  y_train_seq: {y_train_seq.shape}')
print(f'  X_val_seq:   {X_val_seq.shape}    y_val_seq:   {y_val_seq.shape}')
print(f'  X_test_seq:  {X_test_seq.shape}   y_test_seq:  {y_test_seq.shape}')

In [ ]:
# Verify saved splits
from pathlib import Path
import yaml

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

splits_dir = Path('../' + cfg['data']['splits_path'])

X_train     = np.load(splits_dir / 'X_train.npy')
X_val       = np.load(splits_dir / 'X_val.npy')
X_test      = np.load(splits_dir / 'X_test.npy')
y_train     = np.load(splits_dir / 'y_train.npy')
y_val       = np.load(splits_dir / 'y_val.npy')
y_test      = np.load(splits_dir / 'y_test.npy')
X_train_seq = np.load(splits_dir / 'X_train_seq.npy')
X_val_seq   = np.load(splits_dir / 'X_val_seq.npy')
X_test_seq  = np.load(splits_dir / 'X_test_seq.npy')
y_train_seq = np.load(splits_dir / 'y_train_seq.npy')
y_val_seq   = np.load(splits_dir / 'y_val_seq.npy')
y_test_seq  = np.load(splits_dir / 'y_test_seq.npy')

print('=== Flat splits (for Random Forest) ===')
print(f'  X_train:  {X_train.shape}  y_train: {y_train.shape}')
print(f'  X_val:    {X_val.shape}    y_val:   {y_val.shape}')
print(f'  X_test:   {X_test.shape}   y_test:  {y_test.shape}')
print('\n=== Sequence splits (for LSTM/GRU) ===')
print(f'  X_train_seq: {X_train_seq.shape}  y_train_seq: {y_train_seq.shape}')
print(f'  X_val_seq:   {X_val_seq.shape}    y_val_seq:   {y_val_seq.shape}')
print(f'  X_test_seq:  {X_test_seq.shape}   y_test_seq:  {y_test_seq.shape}')

In [ ]:
# Verify label balance across splits
for name, y in [('train', y_train), ('val', y_val), ('test', y_test)]:
    n_normal = (y == 0).sum()
    n_ddos   = (y == 1).sum()
    print(f'{name}: normal={n_normal:,} ({n_normal/len(y)*100:.1f}%)  '
          f'ddos={n_ddos:,} ({n_ddos/len(y)*100:.1f}%)')

In [ ]:
# Verify scaler was fit correctly (train mean ~0, std ~1)
import joblib
scaler = joblib.load(splits_dir / 'scaler.pkl')
print('Scaler mean (first 5 features):', scaler.mean_[:5].round(4))
print('Scaler std  (first 5 features):', scaler.scale_[:5].round(4))
print('X_train mean (should be ~0):',    X_train.mean(axis=0)[:5].round(4))
print('X_train std  (should be ~1):',    X_train.std(axis=0)[:5].round(4))